# EWMA Volatility Inference with VIX

**Optimale Konfiguration (aus Ablation Study):**
- K=2 Features: `ewma_vol_lag_1` + `vix_lag_1`
- MAE Verbesserung: **11.3%** gegenüber K=1 ohne VIX
- R²: **94.3%**

VIX ("Fear Index") misst die erwartete Marktvolatilität und ergänzt die glatte EWMA-Kurve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from chronos import Chronos2Pipeline

## Configuration

In [ ]:
# Paths
csv_path = "../processed_data/data_UBS_UBS.csv"
VIX_PATH = "../data_fred/vixcls.csv"

# Column settings
date_col = "timestamp"
id_col = "id"
target_col = "ewma_vol"

# OPTIMAL K=2 Features (from ablation study)
feature_cols = ["ewma_vol_lag_1", "vix_lag_1"]

# EWMA Settings
LAMBDA = 0.94  # RiskMetrics standard

# Train/Test Split
split_start_date = "2020-01-01"
train_fraction = 0.80

# Device
device = "cuda"

## EWMA Calculation Function

In [ ]:
def calculate_ewma_volatility(returns, lambda_=0.94):
    """Calculate EWMA (Exponentially Weighted Moving Average) Volatility."""
    n = len(returns)
    if n == 0:
        return pd.Series(dtype=float)
    
    ewma_var = np.zeros(n)
    returns_filled = returns.fillna(0).values
    ewma_var[0] = returns_filled[0] ** 2
    
    for t in range(1, n):
        ewma_var[t] = lambda_ * ewma_var[t-1] + (1 - lambda_) * (returns_filled[t] ** 2)
    
    ewma_vol = np.sqrt(ewma_var)
    return pd.Series(ewma_vol, index=returns.index)

## Load and Prepare Data

In [ ]:
# Load stock data
print("Loading stock data...")
df = pd.read_csv(csv_path)
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True)

if id_col not in df.columns:
    df[id_col] = "series_1"

# Calculate log returns
if "log_return" not in df.columns:
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))

# Calculate EWMA
print(f"Calculating EWMA volatility (λ={LAMBDA})...")
df["ewma_vol"] = calculate_ewma_volatility(df["log_return"], LAMBDA)
df["ewma_vol_lag_1"] = df["ewma_vol"].shift(1)

print(f"Stock data: {len(df)} rows")

In [ ]:
# Load VIX data
print("Loading VIX data...")
vix = pd.read_csv(VIX_PATH)
vix.columns = ['date', 'vix']
vix['date'] = pd.to_datetime(vix['date'])
vix['vix'] = pd.to_numeric(vix['vix'], errors='coerce')
vix = vix.dropna()
vix['vix'] = vix['vix'] / 100  # Normalize
vix['vix_lag_1'] = vix['vix'].shift(1)

# Merge VIX with stock data
df = df.merge(vix[['date', 'vix_lag_1']], left_on=date_col, right_on='date', how='left')
df = df.drop(columns=['date'], errors='ignore')
df = df.dropna()

print(f"VIX merged: {len(df)} rows")
df.head()

## Business Day Reindexing

In [ ]:
# Keep only needed columns
keep_cols = [id_col, date_col, target_col] + feature_cols
df = df[keep_cols].copy()

# Business day reindexing
pieces = []
for sid, g in df.groupby(id_col):
    g = g.sort_values(date_col).drop_duplicates(subset=[date_col])
    g = g.set_index(date_col)
    bdays_idx = pd.date_range(start=g.index.min(), end=g.index.max(), freq="B")
    g = g.reindex(bdays_idx)
    g = g.interpolate().ffill().bfill()
    g[id_col] = sid
    g.index.name = date_col
    pieces.append(g.reset_index())

df = pd.concat(pieces, ignore_index=True).dropna()
print(f"Data ready: {len(df)} rows")

## Train/Test Split

In [ ]:
df = df.sort_values([id_col, date_col]).reset_index(drop=True)
df_split = df[df[date_col] >= pd.Timestamp(split_start_date)]
df_split = df_split.sort_values(date_col).reset_index(drop=True)

n_total = len(df_split)
n_train = int(np.floor(train_fraction * n_total))

context_df = df_split.iloc[:n_train].copy()
future_df = df_split.iloc[n_train:].copy()

print(f"Context: {len(context_df)} rows")
print(f"Future:  {len(future_df)} rows")

## Load Model and Run Inference

In [ ]:
print(f"Loading Chronos-2 on {device}...")
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)
print("Model loaded!")

In [ ]:
pred_len = len(future_df)

context_cols = [id_col, date_col, target_col] + feature_cols
future_cols = [id_col, date_col] + feature_cols

print(f"Running inference for {pred_len} predictions...")

pred_df = pipeline.predict_df(
    context_df[context_cols].reset_index(drop=True),
    future_df=future_df[future_cols].reset_index(drop=True),
    prediction_length=pred_len,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column=id_col,
    timestamp_column=date_col,
    target=target_col
)

print("Inference complete!")

## Calculate Metrics

In [ ]:
results = pred_df.merge(
    future_df[[id_col, date_col, target_col]],
    on=[id_col, date_col],
    how="left"
)

y_true = results[target_col].astype(float)
y_pred = results["0.5"].astype(float)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - len(feature_cols) - 1)

print("="*50)
print("EWMA + VIX VOLATILITY METRICS")
print("="*50)
print(f"Features: {feature_cols}")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")
print(f"Adj R²: {adj_r2:.4f}")

## Plot Forecast

In [ ]:
plt.figure(figsize=(14, 6))

# Historical
hist = context_df[[date_col, target_col]].sort_values(date_col).tail(len(future_df))
plt.plot(hist[date_col], hist[target_col], label="Historical", color='blue', alpha=0.7)

# True
true_f = future_df[[date_col, target_col]].sort_values(date_col)
plt.plot(true_f[date_col], true_f[target_col], label="True", color='green', marker='o', markersize=2)

# Prediction
pred = results[[date_col, "0.5", "0.1", "0.9"]].sort_values(date_col)
plt.plot(pred[date_col], pred["0.5"], label="Prediction", color='red', linestyle='--', marker='x', markersize=3)

# Confidence interval
plt.fill_between(pred[date_col], pred["0.1"], pred["0.9"], alpha=0.2, color='red', label="10-90%")

# Forecast start line
plt.axvline(true_f[date_col].min(), color='gray', linestyle='--', label="Forecast start")

stock = csv_path.split("data_")[1].split("_")[0] if "data_" in csv_path else "Stock"
plt.title(f"EWMA + VIX Volatility Forecast: {stock}\nMAE: {mae:.6f} | R²: {r2:.4f}")
plt.xlabel("Date")
plt.ylabel("EWMA Volatility")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ewma_vix_forecast.png", dpi=150)
plt.show()